# Conversational Multi-Agent (AutoGen-style) | Multi-Agent Collaboration

In [1]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing import TypedDict, List, Literal
from typing_extensions import NotRequired
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
model = ChatOpenAI(model="gpt-4o")

In [4]:
AGENTS = {
    "architect": "You are a software architect. Focus on system design, scalability, and architecture decisions.",
    "developer": "You are a senior developer. Focus on implementation details, code patterns, and practical solutions.",
    "tester": "You are a QA engineer. Focus on testing strategies, edge cases, and potential failure modes.",
}

class ConversationState(TypedDict):
    problem: str
    messages: List[str]
    turn: NotRequired[int]
    solution: NotRequired[str]

MAX_TURNS = 6  # 2 rounds of 3 agents

In [5]:
def select_next_speaker(state: ConversationState) -> str:
    """LLM-driven speaker selection — decide who should respond based on conversation context."""
    messages = state.get("messages", [])
    if not messages:
        return "architect"  # Start with the architect for system design problems
    conversation_history = "\n\n".join(messages[-3:])  # Last 3 messages for context
    response = model.invoke(
        f"You are a group chat moderator. Based on the conversation below, decide which agent "
        f"should speak NEXT. Choose the agent whose expertise is most needed right now.\n\n"
        f"Available agents: architect (system design), developer (implementation), tester (QA/testing)\n\n"
        f"Recent conversation:\n{conversation_history}\n\n"
        f"Reply with ONLY the agent name: architect, developer, or tester"
    )
    choice = response.content.strip().lower()
    valid = {"architect", "developer", "tester"}
    return choice if choice in valid else list(AGENTS.keys())[state.get("turn", 0) % len(AGENTS)]

def check_termination(state: ConversationState) -> bool:
    """Check if the discussion has reached a natural conclusion."""
    messages = state.get("messages", [])
    if len(messages) < 3:
        return False  # Need at least one round before checking
    conversation_history = "\n\n".join(messages[-3:])
    response = model.invoke(
        f"Has this technical discussion reached a clear, actionable conclusion? "
        f"Consider: Are there open questions? Do agents agree on a solution?\n\n"
        f"Recent messages:\n{conversation_history}\n\n"
        f"Reply with ONLY: YES or NO"
    )
    return response.content.strip().upper().startswith("YES")

consecutive_done = 0  # Track consecutive YES termination checks

def agent_speak(state: ConversationState) -> Command[Literal["agent_speak", "synthesize"]]:
    """Current agent contributes, then LLM selects next speaker and checks termination."""
    global consecutive_done
    turn = state.get("turn", 0)
    speaker = select_next_speaker(state)
    system_prompt = AGENTS[speaker]

    conversation_history = "\n\n".join(state.get("messages", []))
    response = model.invoke(
        f"{system_prompt}\n\n"
        f"Problem: {state['problem']}\n\n"
        f"Conversation so far:\n{conversation_history or '(You are starting the discussion.)'}\n\n"
        f"Provide your perspective. Build on what others have said. "
        f"Be specific and actionable. Keep your response focused (2-3 paragraphs max)."
    )
    new_message = f"**{speaker.title()}**: {response.content}"
    messages = list(state.get("messages", [])) + [new_message]
    new_turn = turn + 1
    update = {"messages": messages, "turn": new_turn}
    # Check for early termination: 2 consecutive YES = discussion concluded
    if new_turn >= 3 and check_termination({"messages": messages}):
        consecutive_done += 1
        if consecutive_done >= 2:
            print(f"[Turn {new_turn}] Discussion concluded early (2 consecutive termination signals)")
            return Command(goto="synthesize", update=update)
    else:
        consecutive_done = 0
    if new_turn >= MAX_TURNS:
        return Command(goto="synthesize", update=update)
    return Command(goto="agent_speak", update=update)

def synthesize(state: ConversationState) -> dict:
    """Synthesize the conversation into a final solution."""
    conversation = "\n\n".join(state["messages"])
    response = model.invoke(
        f"Synthesize this multi-agent discussion into a clear, actionable solution.\n\n"
        f"Problem: {state['problem']}\n\n"
        f"Discussion:\n{conversation}\n\n"
        f"Provide a structured solution that incorporates the best ideas from all participants."
    )
    return {"solution": response.content}

In [6]:
graph = StateGraph(ConversationState)
graph.add_node("agent_speak", agent_speak, destinations=("agent_speak", "synthesize"))
graph.add_node("synthesize", synthesize)

graph.add_edge(START, "agent_speak")
# agent_speak returns Command to route directly
graph.add_edge("synthesize", END)

group_chat = graph.compile()

In [7]:
# Plot the workflow
plot_mermaid(group_chat)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	agent_speak(agent_speak)
	synthesize(synthesize)
	__end__([<p>__end__</p>]):::last
	__start__ --> agent_speak;
	agent_speak -.-> synthesize;
	synthesize --> __end__;
	agent_speak -.-> agent_speak;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [8]:
result = group_chat.invoke({
    "problem": "Design a real-time notification system that needs to handle 1M+ concurrent users, "
               "support multiple channels (push, email, SMS), and allow user preference management.",
    "messages": [],
})
print("=== Conversation ===")
for msg in result["messages"]:
    print(f"\n{msg}\n{'---'}")
print(f"\n=== Final Solution ===\n{result['solution']}")

=== Conversation ===

**Architect**: To design a real-time notification system capable of handling over 1 million concurrent users, we need to prioritize scalability, fault tolerance, and support for multiple communication channels. First, consider a distributed microservices architecture, where each service is responsible for a specific function: user preference management, notification delivery, and integration with third-party services (like email or SMS providers). This approach facilitates independent scaling and allows you to allocate resources based on the demand for each service.

For real-time processing, implementing a message broker such as Apache Kafka or RabbitMQ will be crucial. These tools can handle high-throughput messaging and decouple the production of notifications from their consumption, allowing notification delivery services to process them asynchronously and efficiently. Additionally, using pub/sub mechanisms within these brokers can help manage distribution to 



**Developer**: To build on the architect's framework, let's dive into some specific implementation strategies that address both scalability and performance. First, for user preference management, consider a service-oriented design using RESTful APIs or GraphQL. This service should allow for efficient CRUD operations on user preferences, which are essential for personalizing notifications. Implement rate limiting and load balancing at the API gateway level using tools like NGINX or HAProxy to prevent bottlenecks and ensure users receive a fast, consistent experience. For caching, Redis can serve as both a cache and message broker in some real-time applications, reducing latency for preference lookups and storing transient states.

Regarding the notification delivery system, use Kafka for its high throughput and resilience, coupled with a reactive programming model using frameworks like Spring WebFlux or Node.js with streams, which are well-suited for handling asynchronous I/O operatio

In [9]:
stream_invoke(group_chat, {
    "problem": "Design a real-time notification system that needs to handle 1M+ concurrent users, "
               "support multiple channels (push, email, SMS), and allow user preference management.",
    "messages": [],
})


────────────────────────────────────────────────────────────────────────────────
  STREAMING EXECUTION
────────────────────────────────────────────────────────────────────────────────

┌─ UNKNOWN
└────────────────────────────────────────

┌─ UNKNOWN
└────────────────────────────────────────

┌─ UNKNOWN
└────────────────────────────────────────

┌─ UNKNOWN
└────────────────────────────────────────

┌─ UNKNOWN
└────────────────────────────────────────

┌─ UNKNOWN
└────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────
  EXECUTION COMPLETE
────────────────────────────────────────────────────────────────────────────────



{'problem': 'Design a real-time notification system that needs to handle 1M+ concurrent users, support multiple channels (push, email, SMS), and allow user preference management.',
 'messages': ['**Architect**: To design a real-time notification system capable of handling over 1 million concurrent users, we need to prioritize scalability, fault tolerance, and support for multiple communication channels. First, consider a distributed microservices architecture, where each service is responsible for a specific function: user preference management, notification delivery, and integration with third-party services (like email or SMS providers). This approach facilitates independent scaling and allows you to allocate resources based on the demand for each service.\n\nFor real-time processing, implementing a message broker such as Apache Kafka or RabbitMQ will be crucial. These tools can handle high-throughput messaging and decouple the production of notifications from their consumption, allo